# Phase 4 — Model training dan persistence
Model dilatih melalui `python -m src.train` menggunakan artifact grouped_v1. Notebook menampilkan hasil training yang sudah tersimpan, tanpa training ulang. PRD.md menjadi acuan; hyperparameter baseline dibekukan sebelum prediksi test.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'PRD.md').exists():
    ROOT = ROOT.parent
if not (ROOT / 'PRD.md').exists():
    raise RuntimeError('Buka notebook dari root proyek atau direktori notebooks')
sys.path.insert(0, str(ROOT))
from src.audit import describe_dataset


In [2]:
plan = json.loads((ROOT / 'reports/grouped_v1/model_plan.json').read_text(encoding='utf-8'))
print(json.dumps(plan, indent=2))


{
  "created_utc": "2026-11-25T16:16:14.705589+00:00",
  "instructions": "PRD.md",
  "configuration_basis": "Fixed baseline chosen before test predictions; PRD contains no numerical hyperparameters.",
  "selection_metric": "f1_macro",
  "test_usage": "Final evaluation only; no tuning or early stopping.",
  "imbalance": "LR/SVM balanced weights computed from training labels; NB uses empirical training class priors.",
  "models": {
    "naive_bayes": {
      "alpha": 1.0,
      "class_prior": null,
      "fit_prior": true,
      "force_alpha": true
    },
    "logistic_regression": {
      "C": 1.0,
      "class_weight": "balanced",
      "dual": false,
      "fit_intercept": true,
      "intercept_scaling": 1,
      "l1_ratio": 0.0,
      "max_iter": 2000,
      "n_jobs": null,
      "penalty": "deprecated",
      "random_state": 42,
      "solver": "lbfgs",
      "tol": 0.0001,
      "verbose": 0,
      "warm_start": false
    },
    "linear_svm": {
      "C": 1.0,
      "class_weight"

In [3]:
training = json.loads((ROOT / 'models/grouped_v1/training_manifest.json').read_text(encoding='utf-8'))
for name, info in training['models'].items():
    print(name, 'seconds=', round(info['training_seconds'], 3),
          'n_iter=', info['n_iter'], 'warnings=', info['warnings'],
          'reload_identical=', info['reload_training_predictions_identical'])


naive_bayes seconds= 0.085 n_iter= None warnings= [] reload_identical= True
logistic_regression seconds= 5.084 n_iter= [82] warnings= [] reload_identical= True
linear_svm seconds= 9.262 n_iter= 17 warnings= [] reload_identical= True


## Kebijakan eksperimen
Ketiga model memakai matriks dan label training yang sama. LR dan SVM memakai class_weight=balanced yang dihitung dari training. Naive Bayes memakai prior empiris. Tidak ada tuning memakai test. Model Joblib dilengkapi checksum, konfigurasi, dan versi environment.
